In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import pickle

# SMOTE
from imblearn.over_sampling import SMOTE

df = pd.read_csv('combined_with_sentiment.csv')

# Ensure target is string (important for stratify + SMOTE)
df['recommended'] = df['recommended'].astype(str)

# Fill missing review text
df['cleaned_review_text'] = df['cleaned_review_text'].fillna('')

# Text + Target
X_text = df['cleaned_review_text']
y = df['recommended']

# Numeric columns
numeric_cols = [
    'neg', 'neu', 'pos', 'compound',
]

# Fill numeric missing
df[numeric_cols] = df[numeric_cols].fillna(0)

X_numeric = df[numeric_cols]

# Split — correct order
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text,
    X_numeric,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Explicitly ensure correct type
y_train = y_train.astype(str)
y_test = y_test.astype(str)

# TF-IDF (fit on train only)
tfidf = TfidfVectorizer(
    max_features=20000, #attempt increase to 10k
    ngram_range=(1,3), # increase to 3
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

# Combine text + numeric into one matrix
X_train = hstack([X_train_tfidf, X_num_train.values])
X_test = hstack([X_test_tfidf, X_num_test.values])

# APPLY SMOTE (only on training) 
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

print("After SMOTE class distribution:\n")
print(y_train.value_counts())

# Save final output
with open('tfidf_split.pkl', 'wb') as f:
    pickle.dump((X_train, X_test, y_train, y_test, tfidf, numeric_cols), f)

print("Shapes:")
print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("\nTrain class distribution:\n", y_train.value_counts())
print("\nTest class distribution:\n", y_test.value_counts())


After SMOTE class distribution:

recommended
yes    13554
no     13554
Name: count, dtype: int64
Shapes:
Train: (27108, 20004)
Test : (4338, 20004)

Train class distribution:
 recommended
yes    13554
no     13554
Name: count, dtype: int64

Test class distribution:
 recommended
no     3389
yes     949
Name: count, dtype: int64
